# Exploration of the data and some algorithms

## Imports

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import numpy as np

from service.datasetservice.DatasetService import DatasetService
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score, ConfusionMatrixDisplay, roc_curve, auc, RocCurveDisplay

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

## Dataset Analysis

In [2]:
dataset_service = DatasetService()
dataset = dataset_service.load_dataset()
dataset.describe()

In [3]:
dataset.head(10)

### Class balances

In [4]:
dataset["Label"].value_counts().plot(kind="barh")

## Data Preparation

### Drop NaN

In [5]:
dataset = dataset_service.remove_nan(dataset=dataset)
dataset.describe()

### Get train and test data

In [6]:
x, y, labels = dataset_service.get_features_and_labels(dataset=dataset)

#### Oversampling minority class

In [13]:
# Shuffle to remove time construct
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2, shuffle=True, random_state=42)

In [16]:
print(np.count_nonzero(train_y==0))

In [24]:
x_resampled, y_resampled = SMOTE().fit_resample(X=x, y=y)

print("Before oversampling")
print("No Stress: ", np.count_nonzero(y==0))
print("Stress: ", np.count_nonzero(y==1))
print("\nAfter oversampling")
print("No Stress: ", np.count_nonzero(y_resampled==0))
print("Stress: ", np.count_nonzero(y_resampled==0))

## Model

In [26]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# model = LogisticRegression(random_state=42).fit(X=train_x, y=train_y)
# model = XGBClassifier(n_estimators=50, max_depth=50, learning_rate=1, objective='binary:logistic', random_state=42).fit(X=train_x, y=train_y)
# grid search
model = XGBClassifier()
n_estimators = [50, 100, 150, 200]
max_depth = [2, 4, 6, 8]
print(max_depth)
param_grid = dict(max_depth=max_depth, n_estimators=n_estimators)
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=7)
grid_search = GridSearchCV(model, param_grid, scoring="neg_log_loss", n_jobs=-1, cv=kfold, verbose=1)
grid_result = grid_search.fit(train_x, train_y)
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

In [27]:
pred = grid_result.best_estimator_.predict(X=test_x)

### Scores

In [28]:
cm = confusion_matrix(y_true=test_y, y_pred=pred)
acc = accuracy_score(pred, test_y)
rec = recall_score(pred, test_y)
prec = precision_score(pred, test_y)
f1 = f1_score(pred, test_y)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()

In [29]:
print("Accuracy: ", acc)
print("Recall: ", rec)
print("Precision: ", prec)
print("F1 Score: ", f1)

In [30]:
disp = RocCurveDisplay.from_estimator(estimator=grid_result.best_estimator_, X=test_x, y=test_y)